<a id="mlp-forward"></a>

$$
\text{多层感知机 (Multi-Layer Perceptron, MLP) 的前向传播 (Forward Pass)} \\ \quad \\
\text{一个隐藏层的二分类网络 (n 个样本按行堆叠, } X \in \mathbb{R}^{n \times d} \text{, 隐藏层 h 个神经元):} \\
Z_1 = X W_1 + b_1 \quad (n \times h) \\
A_1 = \tanh(Z_1) \quad (\text{激活函数, 逐元素作用}) \\
z_2 = A_1 W_2 + b_2 \quad (n \times 1) \\
p = \sigma(z_2) \quad (\text{输出属于类别 1 的概率, 就是对隐藏层特征做逻辑回归}) \\
L = -\frac{1}{n} \sum_i \left[ y_i \ln p_i + (1 - y_i) \ln (1 - p_i) \right] \\ \quad \\
\text{为什么需要非线性激活函数: 如果去掉 tanh,} \\
z_2 = (X W_1 + b_1) W_2 + b_2 = X \underbrace{(W_1 W_2)}_{W} + \underbrace{(b_1 W_2 + b_2)}_{b} \\
\text{多层线性变换的复合还是线性变换, 网络退化成逻辑回归, 只能画出直线边界} \\ \quad \\
\text{tanh 的导数: } \tanh x = \frac{e^x - e^{-x}}{e^x + e^{-x}} = \frac{1 - e^{-2x}}{1 + e^{-2x}} = 2\sigma(2x) - 1 \\
\tanh'(x) = 2 \cdot \sigma'(2x) \cdot 2 = 4 \sigma(2x)(1 - \sigma(2x)) \\
\text{记 } s = \sigma(2x) = \frac{1 + \tanh x}{2} \text{: } 4 s (1 - s) = (1 + \tanh x)(1 - \tanh x) = 1 - \tanh^2 x
$$

数学基础: [链式法则](../math_basics/calculus.ipynb#derivative-rules) · [Sigmoid 及其导数](sigmoid_function.ipynb) · [逻辑回归](logistic_regression.ipynb#logistic-model)

<a id="backprop"></a>

$$
\text{反向传播 (Backpropagation): 用链式法则从输出层往回逐层求梯度} \\
\text{记 } \delta = \frac{\partial L}{\partial z} \text{ (损失对某一层 "激活前" 的值的梯度, 叫误差项)} \\ \quad \\
\text{1. 输出层: 由逻辑回归的推导, 每个样本 } \frac{\partial \ell_i}{\partial z_{2,i}} = p_i - y_i \text{, 平均损失要再除以 n} \\
\delta_2 = \frac{\partial L}{\partial z_2} = \frac{1}{n} (p - y) \quad (n \times 1) \\ \quad \\
\text{2. 输出层参数: } z_{2,i} = \sum_k A_{1,ik} W_{2,k} + b_2 \implies \frac{\partial z_{2,i}}{\partial W_{2,k}} = A_{1,ik} \\
\frac{\partial L}{\partial W_{2,k}} = \sum_i \delta_{2,i} A_{1,ik} \implies \frac{\partial L}{\partial W_2} = A_1^T \delta_2, \quad \frac{\partial L}{\partial b_2} = \sum_i \delta_{2,i} \\ \quad \\
\text{3. 传回隐藏层: } \frac{\partial z_{2,i}}{\partial A_{1,ik}} = W_{2,k} \implies \frac{\partial L}{\partial A_1} = \delta_2 W_2^T \quad (n \times h) \\ \quad \\
\text{4. 穿过激活函数: } A_{1,ik} = \tanh(Z_{1,ik}) \text{ 只依赖同一个位置的 } Z_{1,ik} \text{ (Jacobian 是对角矩阵), 所以是逐元素相乘} \\
\delta_1 = \frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1} \odot \tanh'(Z_1) = (\delta_2 W_2^T) \odot (1 - A_1^2) \\ \quad \\
\text{5. 隐藏层参数 (和第 2 步同样的推导): } \frac{\partial L}{\partial W_1} = X^T \delta_1, \quad \frac{\partial L}{\partial b_1} = \sum_i \delta_{1,i} \\ \quad \\
\text{规律: 每一层 } \frac{\partial L}{\partial W} = (\text{这一层的输入})^T \times \delta \text{, 而 } \delta \text{ 从后往前传: 乘以 } W^T \text{, 再逐元素乘以激活函数的导数} \\
\text{这就是 Jacobian 连乘 } \frac{\partial L}{\partial X} = \frac{\partial L}{\partial z_2} J_{z_2 \leftarrow A_1} J_{A_1 \leftarrow Z_1} J_{Z_1 \leftarrow X} \text{, 从左边 (标量损失那一端) 开始乘,} \\
\text{每一步都是 "行向量 × 矩阵", 不需要真的构造巨大的 Jacobian 矩阵}
$$

数学基础: [Jacobian 与链式法则的矩阵形式](../math_basics/calculus.ipynb#jacobian) · [偏导数](../math_basics/calculus.ipynb#partial) · [逻辑回归的梯度 p - y](logistic_regression.ipynb#logistic-gradient)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x_t = np.linspace(-3, 3, 7)
h = 1e-6
print(f"tanh = 2 sigmoid(2x) - 1 ? {np.allclose(np.tanh(x_t), 2 * sigmoid(2 * x_t) - 1)}, "
      f"tanh' = 1 - tanh^2 ? {np.allclose((np.tanh(x_t + h) - np.tanh(x_t - h)) / (2 * h), 1 - np.tanh(x_t)**2)}")

def init_params(d, n_hidden, rng):
    # Xavier 初始化, 见最后一节的推导
    return {'W1': rng.normal(0, np.sqrt(1 / d), size=(d, n_hidden)), 'b1': np.zeros(n_hidden),
            'W2': rng.normal(0, np.sqrt(1 / n_hidden), size=(n_hidden, 1)), 'b2': np.zeros(1)}

def forward(params, X):
    Z1 = X @ params['W1'] + params['b1']
    A1 = np.tanh(Z1)
    z2 = A1 @ params['W2'] + params['b2']
    p = sigmoid(z2).ravel()
    return p, {'A1': A1}

def loss_fn(params, X, y):
    p, _ = forward(params, X)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def backward(params, X, y):
    n = len(y)
    p, cache = forward(params, X)
    A1 = cache['A1']
    delta2 = ((p - y) / n)[:, None]                  # n x 1
    grads = {'W2': A1.T @ delta2, 'b2': delta2.sum(axis=0)}
    dA1 = delta2 @ params['W2'].T                    # n x h
    delta1 = dA1 * (1 - A1**2)                       # 逐元素乘 tanh'
    grads['W1'] = X.T @ delta1
    grads['b1'] = delta1.sum(axis=0)
    return grads

<a id="gradient-check"></a>

$$
\text{梯度检验 (Gradient Check): 反向传播的代码很容易写错, 用数值差分逐个参数验证} \\
\frac{\partial L}{\partial \theta_k} \approx \frac{L(\theta + h e_k) - L(\theta - h e_k)}{2h} \quad (\text{中心差分, 误差 } O(h^2)) \\
\text{相对误差 } \frac{|g_{bp} - g_{num}|}{|g_{bp}| + |g_{num}|} \text{ 在 } 10^{-7} \text{ 以下说明推导和代码都正确}
$$

数学基础: [数值微分与误差阶](../math_basics/calculus.ipynb#numerical-diff)

In [ ]:
rng = np.random.default_rng(0)
X_small = rng.normal(size=(8, 2))
y_small = (rng.random(8) > 0.5).astype(float)
params = init_params(2, 5, rng)
grads = backward(params, X_small, y_small)

for name in ['W1', 'b1', 'W2', 'b2']:
    num = np.zeros_like(params[name])
    for idx in np.ndindex(params[name].shape):
        old = params[name][idx]
        params[name][idx] = old + h; L_plus = loss_fn(params, X_small, y_small)
        params[name][idx] = old - h; L_minus = loss_fn(params, X_small, y_small)
        params[name][idx] = old
        num[idx] = (L_plus - L_minus) / (2 * h)
    rel = np.abs(grads[name] - num).max() / (np.abs(grads[name]).max() + np.abs(num).max())
    print(f'{name}: 反向传播 vs 数值差分, 最大相对误差 = {rel:.2e}')

<a id="mlp-training"></a>

$$
\text{训练: 梯度下降 } \theta \leftarrow \theta - \eta \frac{\partial L}{\partial \theta} \text{ (对所有参数)} \\
\text{和逻辑回归不同, 神经网络的损失不是凸函数 (隐藏层的参数互相交换位置, 损失不变, 存在很多等价的最优点)} \\
\text{所以结果依赖初始化, 但实践中梯度下降通常能找到足够好的解} \\ \quad \\
\text{数据: 两个交错的半月形 (moons), 线性模型分不开}
$$

In [ ]:
X, y = make_moons(n_samples=400, noise=0.2, random_state=0)
X_train, y_train, X_test, y_test = X[:300], y[:300], X[300:], y[300:]

rng = np.random.default_rng(1)
params = init_params(2, 16, rng)
eta = 1.0
losses = []
for epoch in range(6000):
    grads = backward(params, X_train, y_train)
    for name in params:
        params[name] -= eta * grads[name]
    if epoch % 50 == 0:
        losses.append(loss_fn(params, X_train, y_train))

acc = lambda p, y: np.mean((p >= 0.5) == y)
p_train, _ = forward(params, X_train)
p_test, _ = forward(params, X_test)
print(f'手写 MLP (16 个隐藏神经元): 训练准确率 = {acc(p_train, y_train):.4f}, 测试准确率 = {acc(p_test, y_test):.4f}')

sk_mlp = MLPClassifier(hidden_layer_sizes=(16,), activation='tanh', solver='lbfgs', alpha=0, max_iter=5000, random_state=0).fit(X_train, y_train)
print(f'sklearn MLPClassifier       : 训练准确率 = {sk_mlp.score(X_train, y_train):.4f}, 测试准确率 = {sk_mlp.score(X_test, y_test):.4f}')
lr = LogisticRegression().fit(X_train, y_train)
print(f'逻辑回归 (没有隐藏层)       : 训练准确率 = {lr.score(X_train, y_train):.4f}, 测试准确率 = {lr.score(X_test, y_test):.4f}')

g1, g2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])
p_grid, _ = forward(params, grid)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(np.arange(len(losses)) * 50, losses, label='train loss')
axs[0].set_xlabel('epoch')
axs[1].contourf(g1, g2, p_grid.reshape(g1.shape), levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.4)
axs[1].contour(g1, g2, p_grid.reshape(g1.shape), levels=[0.5], colors='k')
axs[1].scatter(*X_train[y_train == 0].T, s=6, c='b', label='class 0')
axs[1].scatter(*X_train[y_train == 1].T, s=6, c='r', label='class 1')
plot_nicely()
plt.show()

<a id="initialization"></a>

$$
\text{初始化 (Initialization)} \\ \quad \\
\text{1. 不能全部初始化为 0 (或相同的值): 对称性 (symmetry)} \\
\text{如果 } W_1 \text{ 的每一列都相同, 隐藏层每个神经元的输出 } A_{1,ik} \text{ 都相同; } W_2 \text{ 的各个分量也相同时} \\
\delta_1 = (\delta_2 W_2^T) \odot (1 - A_1^2) \text{ 的每一列相同} \implies X^T \delta_1 \text{ 的每一列相同} \\
\text{每次更新后各列仍然相同, h 个神经元永远学到同一个东西, 等价于只有 1 个神经元} \\ \quad \\
\text{2. 随机初始化的尺度 (Xavier / Glorot): 希望每一层输出的方差和输入差不多, 信号不会逐层放大或消失} \\
z = \sum_{j=1}^d w_j x_j \text{, w 和 x 独立且均值为 0:} \\
Var(w_j x_j) = E[w_j^2 x_j^2] - (E[w_j] E[x_j])^2 = E[w_j^2] E[x_j^2] = Var(w) Var(x) \\
Var(z) = \sum_j Var(w_j x_j) = d \cdot Var(w) \cdot Var(x) \\
\text{要让 } Var(z) = Var(x) \implies Var(w) = \frac{1}{d} \quad (\text{d 是这一层的输入个数 fan-in})
$$

数学基础: [期望和方差的性质 (独立变量的方差相加)](../math_basics/statistics.ipynb#expectation-variance-properties)

In [ ]:
# 1. 对称性: 相同的初始化 -> 训练后各个隐藏神经元仍然完全相同
sym = {'W1': np.full((2, 4), 0.3), 'b1': np.zeros(4), 'W2': np.full((4, 1), 0.3), 'b2': np.zeros(1)}
for _ in range(500):
    g = backward(sym, X_train, y_train)
    for name in sym:
        sym[name] -= 1.0 * g[name]
print(f'相同初始化训练 500 步后 W1 = \n{sym["W1"]}\n各列完全相同? {np.allclose(sym["W1"], sym["W1"][:, [0]])}')

# 2. 方差传递: 10 层线性层, 不同的初始化尺度
rng = np.random.default_rng(3)
width = 256
x_signal = rng.normal(size=(1000, width))
for scale_name, std in [('Var(w) = 1/d (Xavier)', np.sqrt(1 / width)), ('std = 0.01 (太小)', 0.01), ('std = 0.1 (太大)', 0.1)]:
    a = x_signal
    variances = []
    for layer in range(10):
        a = a @ rng.normal(0, std, size=(width, width))
        variances.append(a.var())
    print(f'{scale_name:<22}: 第 1, 4, 7, 10 层输出方差 = ' + ', '.join(f'{v:.3e}' for v in variances[::3]))